# Disk Space Monitor & Alert System
**Author:** Mahitha Madhava Das  
**Date:** May 2026  
**Tools:** Python, PowerShell, subprocess, os  

## About This Project
This script monitors disk space across all drives on a Windows machine,
flags drives exceeding a defined threshold, generates a timestamped
alert report, and logs every run to a persistent history file.

This replicates the proactive infrastructure monitoring I performed
at Cognizant Technology Solutions when supporting Liberty Mutual
Insurance — where disk space exhaustion on application servers
could cause batch job failures and application outages.

## Skills Demonstrated
- Python scripting and automation
- Proactive monitoring and threshold alerting
- Incident report generation
- Windows system administration
- Production support workflows

In [1]:
# Import required libraries
import subprocess
import datetime
import os
import platform

# Configuration — change these thresholds to match your needs
WARNING_THRESHOLD  = 70   # Warn if drive is more than 70% full
CRITICAL_THRESHOLD = 85   # Critical alert if drive is more than 85% full
ALERT_LOG_FILE     = "disk_alert_history.log"

print(f"Disk Space Monitor initialized")
print(f"Warning threshold  : {WARNING_THRESHOLD}%")
print(f"Critical threshold : {CRITICAL_THRESHOLD}%")
print(f"Running on         : {platform.system()} {platform.release()}")
print(f"Timestamp          : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

Disk Space Monitor initialized
Warning threshold  : 70%
Critical threshold : 85%
Running on         : Windows 11
Timestamp          : 2026-05-02 15:46:43


In [2]:
def get_disk_usage():
    """
    Query all drives on the system and return usage statistics.
    Uses PowerShell to get accurate Windows drive information.
    
    Returns:
        List of dictionaries with drive details
    """
    
    cmd = """
    Get-PSDrive -PSProvider FileSystem | 
    Where-Object { $_.Used -ne $null } |
    Select-Object Name, 
        @{N='TotalGB';E={[math]::Round(($_.Used + $_.Free)/1GB, 2)}},
        @{N='UsedGB';E={[math]::Round($_.Used/1GB, 2)}},
        @{N='FreeGB';E={[math]::Round($_.Free/1GB, 2)}},
        @{N='UsedPct';E={[math]::Round(($_.Used/($_.Used+$_.Free))*100, 1)}} |
    ConvertTo-Csv -NoTypeInformation
    """
    
    try:
        result = subprocess.run(
            ["powershell", "-Command", cmd],
            capture_output=True, text=True, timeout=30
        )
        
        drives = []
        lines = result.stdout.strip().split('\n')
        
        # Skip header row, parse each drive
        for line in lines[1:]:
            line = line.strip().replace('"', '')
            if not line:
                continue
            parts = line.split(',')
            if len(parts) == 5:
                try:
                    drives.append({
                        'name'     : parts[0] + ":\\",
                        'total_gb' : float(parts[1]),
                        'used_gb'  : float(parts[2]),
                        'free_gb'  : float(parts[3]),
                        'used_pct' : float(parts[4])
                    })
                except ValueError:
                    continue
        
        return drives
    
    except Exception as e:
        print(f"Error querying disk usage: {str(e)}")
        return []

print("Disk check function defined successfully!")

Disk check function defined successfully!


In [3]:
def classify_drive(used_pct):
    """
    Classify a drive's status based on usage percentage.
    Mirrors the severity levels used in ServiceNow incident management.
    
    Returns:
        Tuple of (status_label, urgency)
    """
    if used_pct >= CRITICAL_THRESHOLD:
        return ("CRITICAL", "P1 - Immediate action required")
    elif used_pct >= WARNING_THRESHOLD:
        return ("WARNING",  "P2 - Investigate within 4 hours")
    elif used_pct >= 50:
        return ("MODERATE", "P3 - Monitor closely")
    else:
        return ("HEALTHY",  "No action required")

def draw_bar(used_pct, width=30):
    """
    Draw a simple ASCII progress bar showing disk usage.
    Makes the report visually scannable at a glance.
    """
    filled = int((used_pct / 100) * width)
    bar    = "#" * filled + "-" * (width - filled)
    return f"[{bar}] {used_pct}%"

print("Classification functions defined!")
print()

# Quick test of the bar chart
print("Sample usage bars:")
for pct in [15, 45, 72, 88, 95]:
    status, _ = classify_drive(pct)
    print(f"  {draw_bar(pct)}  {status}")

Classification functions defined!

Sample usage bars:
  [####--------------------------] 15%  HEALTHY
  [#############-----------------] 45%  HEALTHY
  [#####################---------] 72%  WARNING
  [##########################----] 88%  CRITICAL
  [############################--] 95%  CRITICAL


In [4]:
print("=" * 60)
print("        DISK SPACE MONITOR — FULL SCAN")
print("=" * 60)
print(f"Scan time: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# Get all drives
drives = get_disk_usage()

if not drives:
    print("No drives found or error querying system.")
else:
    alerts    = []
    all_clear = True
    
    for drive in drives:
        status, urgency = classify_drive(drive['used_pct'])
        bar = draw_bar(drive['used_pct'])
        
        # Print drive details
        print(f"Drive : {drive['name']}")
        print(f"Usage : {bar}")
        print(f"Space : {drive['used_gb']} GB used  /  "
              f"{drive['free_gb']} GB free  /  "
              f"{drive['total_gb']} GB total")
        print(f"Status: {status}")
        
        # Flag anything needing attention
        if status in ("CRITICAL", "WARNING"):
            all_clear = False
            alerts.append({
                'drive'   : drive['name'],
                'status'  : status,
                'urgency' : urgency,
                'used_pct': drive['used_pct'],
                'free_gb' : drive['free_gb']
            })
            print(f"ACTION: {urgency}")
        
        print("-" * 60)
    
    # Final summary
    print()
    if all_clear:
        print("OVERALL STATUS: ALL CLEAR")
        print("All drives within acceptable thresholds.")
    else:
        print(f"OVERALL STATUS: {len(alerts)} DRIVE(S) NEED ATTENTION")
        print()
        print("ALERTS SUMMARY:")
        for alert in alerts:
            print(f"  {alert['status']} -- Drive {alert['drive']} "
                  f"is {alert['used_pct']}% full "
                  f"({alert['free_gb']} GB remaining)")
            print(f"  Urgency: {alert['urgency']}")

        DISK SPACE MONITOR — FULL SCAN
Scan time: 2026-05-02 15:47:41

Drive : C:\
Usage : [###########################---] 93.0%
Space : 302.47 GB used  /  22.75 GB free  /  325.22 GB total
Status: CRITICAL
ACTION: P1 - Immediate action required
------------------------------------------------------------
Drive : D:\
Usage : [##########################----] 89.1%
Space : 133.58 GB used  /  16.42 GB free  /  150.0 GB total
Status: CRITICAL
ACTION: P1 - Immediate action required
------------------------------------------------------------

OVERALL STATUS: 2 DRIVE(S) NEED ATTENTION

ALERTS SUMMARY:
  CRITICAL -- Drive C:\ is 93.0% full (22.75 GB remaining)
  Urgency: P1 - Immediate action required
  CRITICAL -- Drive D:\ is 89.1% full (16.42 GB remaining)
  Urgency: P1 - Immediate action required


In [5]:
def save_report_and_log(drives):
    """
    Save a timestamped report file AND append to a persistent
    history log so you can track disk trends over time.
    
    In production this history log would feed a dashboard
    showing disk usage trends over days/weeks.
    """
    
    timestamp     = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    file_stamp    = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    report_file   = f"disk_report_{file_stamp}.txt"
    
    # Build report
    lines = []
    lines.append("=" * 60)
    lines.append("     DISK SPACE MONITOR — INCIDENT REPORT")
    lines.append("=" * 60)
    lines.append(f"Generated : {timestamp}")
    lines.append(f"Host      : {os.environ.get('COMPUTERNAME', 'Unknown')}")
    lines.append(f"Thresholds: Warning={WARNING_THRESHOLD}%  "
                 f"Critical={CRITICAL_THRESHOLD}%")
    lines.append("=" * 60)
    lines.append("")
    
    log_entries = []
    
    for drive in drives:
        status, urgency = classify_drive(drive['used_pct'])
        bar = draw_bar(drive['used_pct'])
        
        lines.append(f"Drive  : {drive['name']}")
        lines.append(f"Usage  : {bar}")
        lines.append(f"Space  : {drive['used_gb']}GB used / "
                     f"{drive['free_gb']}GB free / "
                     f"{drive['total_gb']}GB total")
        lines.append(f"Status : {status} -- {urgency}")
        lines.append("")
        
        # One-line log entry for history file
        log_entries.append(
            f"{timestamp} | {drive['name']} | "
            f"{drive['used_pct']}% used | "
            f"{drive['free_gb']}GB free | {status}"
        )
    
    lines.append("=" * 60)
    lines.append("END OF REPORT")
    
    # Save full report
    with open(report_file, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    
    # Append to history log
    with open(ALERT_LOG_FILE, "a", encoding="utf-8") as f:
        f.write("\n".join(log_entries) + "\n")
    
    print(f"Report saved : {report_file}")
    print(f"History log  : {ALERT_LOG_FILE}")
    print(f"Location     : {os.path.abspath(report_file)}")
    return report_file

# Run it
saved = save_report_and_log(drives)

Report saved : disk_report_2026-05-02_15-48-11.txt
History log  : disk_alert_history.log
Location     : C:\Users\dasma\disk_report_2026-05-02_15-48-11.txt


In [6]:
def view_history():
    """
    Display the persistent history log.
    In production this would be a 30-day rolling window
    feeding a Grafana or Splunk dashboard.
    """
    
    print("=" * 60)
    print("     DISK USAGE HISTORY LOG")
    print("=" * 60)
    
    if not os.path.exists(ALERT_LOG_FILE):
        print("No history found. Run the monitor at least once.")
        return
    
    with open(ALERT_LOG_FILE, "r") as f:
        entries = f.readlines()
    
    if not entries:
        print("History log is empty.")
        return
    
    print(f"Total entries: {len(entries)}")
    print()
    
    for entry in entries:
        entry = entry.strip()
        if not entry:
            continue
        # Highlight alerts
        if "CRITICAL" in entry:
            print(f"  *** CRITICAL *** {entry}")
        elif "WARNING" in entry:
            print(f"  !!  WARNING  !!  {entry}")
        else:
            print(f"      OK           {entry}")
    
    print("=" * 60)

view_history()

     DISK USAGE HISTORY LOG
Total entries: 2

  *** CRITICAL *** 2026-05-02 15:48:11 | C:\ | 93.0% used | 22.75GB free | CRITICAL
  *** CRITICAL *** 2026-05-02 15:48:11 | D:\ | 89.1% used | 16.42GB free | CRITICAL


## Results & Observations

### Run environment
- **Date:** May 2, 2026
- **Machine:** Personal Windows laptop (test environment)
- **Thresholds set:** Warning at 70% | Critical at 85%

### Scan findings

| Drive | Used | Free | Total | Status |
|---|---|---|---|---|
| C:\\ | 302.47 GB (93.0%) | 22.75 GB | 325.22 GB | CRITICAL |
| D:\\ | 133.58 GB (89.1%) | 16.42 GB | 150.00 GB | CRITICAL |

**Overall status: CRITICAL — Both drives exceed the 85% threshold**

### Alert summary
- **C:\\** — 93% full, only 22.75 GB remaining — P1 immediate action required
- **D:\\** — 89.1% full, only 16.42 GB remaining — P1 immediate action required

### Real-world significance
Both drives breaching the critical threshold simultaneously is a
realistic production scenario. In an enterprise environment, this
pattern would indicate either:
- A runaway batch job writing logs without rotation
- A large file transfer filling the disk
- A backup process that failed to clean up temp files
- Genuine capacity planning failure requiring disk expansion

### What a real L2 analyst does next

In a production environment, when two drives simultaneously hit
CRITICAL threshold the response would be:

1. **Immediately** open a P1 incident in ServiceNow
2. **Identify** the largest consumers using disk analysis tools
3. **Check** recent batch job logs for abnormal output file sizes
4. **Clear** safe temp files and log archives as a quick fix
5. **Notify** on-call infrastructure team for emergency disk expansion
6. **Post-incident** implement log rotation policy to prevent recurrence

This script detected both issues in a single automated scan —
replacing what would otherwise require manual checks across
every server in the environment.

### Production equivalent
In enterprise environments this script would be:
- Scheduled via **Windows Task Scheduler** to run every hour
- History log feeding a **Splunk dashboard** for trend analysis
- Auto-creating **ServiceNow P1 tickets** when critical threshold breaches
- Alerting on-call engineers via **PagerDuty** within 60 seconds
- Retained as a **30-day rolling log** for capacity planning reviews

### Personal action taken
As a result of running this script, I identified that my own
C: drive (93% full) and D: drive (89.1% full) both require
immediate cleanup — demonstrating that this monitoring tool
provides real, actionable value even outside a production context.

### Why this matters for application support
Disk exhaustion is one of the most common root causes of:
- **Batch job failures** — logs fill the disk mid-run